# Ders 1A — Feature Store Oluşturma

Ham veri setleri okunur, moleküler feature dosyaları üretilir ve sınıf dağılımları görselleştirilir.

## 1. Paket kontrolü

Feature üretimi için gereken paketler kontrol edilir. Model eğitimi yapılmadığı için sklearn import edilmez.

In [ ]:
import sys
# Mevcut Python ortamında pip komutunu çalıştırmak için kullanılır.
import subprocess
# Eksik paketleri notebook içinden kurmak için kullanılır.
import importlib.util
# Bir paketin kurulu olup olmadığını kontrol etmek için kullanılır.

def install_if_missing(import_name, pip_name=None):
    """Eksik paket varsa kurar; paket zaten varsa işlem yapmaz."""
    pip_name = pip_name or import_name
    # pip adı verilmezse import adı paket adı olarak kullanılır.
    if importlib.util.find_spec(import_name) is None:
        # Paket kurulu değilse pip kurulumu yapılır.
        print(f"[INSTALL] {pip_name}")
        # Hangi paketin kurulacağı ekrana yazdırılır.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Paket sessiz modda kurulur.

required_packages = [
    ("pandas", "pandas"),  # CSV okuma ve tablo işlemleri için kullanılır.
    ("numpy", "numpy"),  # Fingerprint matrisleri için kullanılır.
    ("matplotlib", "matplotlib"),  # Sınıf dağılımı grafiği için kullanılır.
]
# Bu scriptte gerçekten kullanılan paketler listelenir.

for import_name, pip_name in required_packages:
    # Paketler tek tek kontrol edilir.
    install_if_missing(import_name, pip_name)
    # Eksik paket varsa kurulur.


if importlib.util.find_spec("rdkit") is None:
    # RDKit moleküler fingerprint ve descriptor üretimi için ayrıca kontrol edilir.
    try:
        # Önce güncel rdkit paketi denenir.
        install_if_missing("rdkit", "rdkit")
    except Exception:
        # İlk yöntem başarısız olursa Colab'da yaygın çalışan alternatif paket denenir.
        install_if_missing("rdkit", "rdkit-pypi")

print("Paket kontrolü tamamlandı.")
# Paket kontrolünün bittiği yazdırılır.

## 2. Importlar ve genel ayarlar

Bu dosyada yalnızca ham veri okuma, RDKit feature üretimi ve grafik için gereken importlar vardır.

In [ ]:
from pathlib import Path
# Dosya ve klasör yollarını güvenli şekilde yönetmek için kullanılır.
import warnings
# Gereksiz uyarıları kontrol etmek için kullanılır.
warnings.filterwarnings("ignore")
# Notebook çıktısını sade tutmak için uyarılar gizlenir.

import numpy as np
# Fingerprint matrisleri ve sayısal işlemler için kullanılır.
import pandas as pd
# Ham CSV dosyalarını okumak ve feature tablolarını kaydetmek için kullanılır.
import matplotlib.pyplot as plt
# Sınıf dağılımı grafiklerini çizmek için kullanılır.

from rdkit import Chem, DataStructs
# SMILES metnini molekül nesnesine çevirmek ve fingerprintleri numpy array'e aktarmak için kullanılır.
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator, Descriptors
# MACCS, Morgan fingerprint ve RDKit descriptor araçları çağırılır.
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator
# RDKit descriptor hesaplayıcısı çağırılır.

DATASETS = {
    # Ham veri linkleri ve çıktı dosyası prefixleri tanımlanır.
    "ERa_BLA_assay": {
        "short_name": "ERα BLA",
        "model_prefix": "model_ERa_BLA",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_1_A14_A17_ERa_BLA_agonist_antagonist.csv",
    },
    "ERa_LUC_VM7_assay": {
        "short_name": "ERα LUC VM7",
        "model_prefix": "model_ERa_LUC_VM7",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_2_A15_A18_ERa_LUC_VM7_agonist_antagonist.csv",
    },
}

TARGET_COLUMN = "binary_label_agonist1_antagonist0"
# Binary sınıf bilgisinin bulunduğu kolon adı tanımlanır.
SMILES_COLUMN = "QSAR-Ready SMILES"
# Molekül SMILES bilgisinin bulunduğu kolon adı tanımlanır.
MORGAN_BITS = 1024
# Morgan fingerprint uzunluğu belirlenir.
MORGAN_RADIUS = 2
# Morgan fingerprint radius değeri belirlenir.

OUTPUT_ROOT = Path("molfest_outputs")
# Grafik ve rapor çıktıları için ana klasör belirlenir.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

feature_store_dir = Path("molfest_outputs/01_feature_store")
# Üretilecek feature CSV dosyalarının klasörü belirlenir.
feature_store_dir.mkdir(parents=True, exist_ok=True)
# Feature store klasörü yoksa oluşturulur.

print("Ayarlar hazır.")
# Ayar hücresinin tamamlandığı yazdırılır.

## 3. Feature üretimi için yardımcı fonksiyonlar

Bu fonksiyonlar SMILES bilgisinden Morgan, MACCS, RDKit descriptor ve varsa Avalon fingerprint üretir.

In [ ]:
def show_table(df, n=50, title=None):
    """Özet tabloları Colab'da okunabilir şekilde gösterir."""
    if title:
        # Başlık varsa tablo öncesinde yazdırılır.
        print("\n" + title)
    try:
        # Colab/Jupyter ortamında tablo display ile gösterilir.
        display(df.head(n))
    except NameError:
        # Terminalde display yoksa tablo metin olarak yazdırılır.
        print(df.head(n).to_string(index=False))


def read_csv_flexible(url):
    """CSV dosyasını önce ';', sonra ',' ayıracıyla okumayı dener."""
    for sep in [";", ","]:
        # Yaygın iki CSV ayıracı sırayla denenir.
        df = pd.read_csv(url, sep=sep, encoding="utf-8-sig", low_memory=False)
        # CSV dosyası ilgili ayraçla okunur.
        if df.shape[1] > 1:
            # Doğru ayraçta birden fazla kolon beklenir.
            return df, sep
            # Okunan tablo ve kullanılan ayraç döndürülür.
    raise ValueError("CSV ayıracı anlaşılamadı.")


def valid_molecules(smiles):
    """SMILES listesini RDKit molecule listesine çevirir."""
    mols, valid = [], []
    # Molekül nesneleri ve geçerlilik bilgileri için listeler oluşturulur.
    for smi in smiles:
        # SMILES değerleri tek tek dolaşılır.
        mol = Chem.MolFromSmiles(str(smi))
        # SMILES metni RDKit molecule nesnesine çevrilir.
        mols.append(mol)
        # Molekül nesnesi listeye eklenir.
        valid.append(mol is not None)
        # Parse başarılıysa True, başarısızsa False kaydedilir.
    return mols, np.array(valid, dtype=bool)
    # Molekül listesi ve geçerlilik maskesi döndürülür.


def smiles_to_morgan(smiles):
    """Morgan fingerprint matrisi üretir."""
    mols, valid = valid_molecules(smiles)
    # SMILES listesi molekül nesnelerine çevrilir.
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_BITS)
    # Morgan fingerprint hesaplayıcısı oluşturulur.
    rows = []
    # Fingerprint satırları burada biriktirilir.
    for mol, keep in zip(mols, valid):
        # Moleküller geçerlilik bilgisiyle birlikte dolaşılır.
        if keep:
            # Geçersiz SMILES atlanır.
            fp = generator.GetFingerprint(mol)
            # Morgan fingerprint hesaplanır.
            arr = np.zeros((MORGAN_BITS,), dtype=np.float32)
            # Fingerprint değerlerini tutacak boş array oluşturulur.
            DataStructs.ConvertToNumpyArray(fp, arr)
            # RDKit fingerprint numpy array'e çevrilir.
            rows.append(arr)
            # Array sonuç listesine eklenir.
    names = [f"Morgan_{i}" for i in range(MORGAN_BITS)]
    # Morgan feature isimleri oluşturulur.
    return np.vstack(rows), names, valid
    # Feature matrisi, feature isimleri ve valid mask döndürülür.


def smiles_to_maccs(smiles):
    """MACCS fingerprint matrisi üretir."""
    mols, valid = valid_molecules(smiles)
    # SMILES listesi molekül nesnelerine çevrilir.
    rows = []
    # MACCS satırları burada biriktirilir.
    for mol, keep in zip(mols, valid):
        # Moleküller geçerlilik bilgisiyle birlikte dolaşılır.
        if keep:
            # Geçersiz SMILES atlanır.
            fp = MACCSkeys.GenMACCSKeys(mol)
            # MACCS fingerprint hesaplanır.
            arr = np.zeros((167,), dtype=np.float32)
            # RDKit MACCS çıktısı için boş array oluşturulur.
            DataStructs.ConvertToNumpyArray(fp, arr)
            # MACCS fingerprint numpy array'e çevrilir.
            rows.append(arr[1:])
            # 0. bit gerçek MACCS key olmadığı için çıkarılır.
    names = [f"MACCS_{i}" for i in range(1, 167)]
    # MACCS feature isimleri oluşturulur.
    return np.vstack(rows), names, valid
    # Feature matrisi, feature isimleri ve valid mask döndürülür.


def smiles_to_rdkit_descriptors(smiles):
    """RDKit descriptor matrisi üretir."""
    mols, valid = valid_molecules(smiles)
    # SMILES listesi molekül nesnelerine çevrilir.
    descriptor_names = [name for name, _ in Descriptors._descList]
    # RDKit descriptor isimleri alınır.
    calc = MolecularDescriptorCalculator(descriptor_names)
    # Descriptor hesaplayıcı oluşturulur.
    rows = []
    # Descriptor satırları burada biriktirilir.
    for mol, keep in zip(mols, valid):
        # Moleküller geçerlilik bilgisiyle birlikte dolaşılır.
        if keep:
            # Geçersiz SMILES atlanır.
            desc = np.array(calc.CalcDescriptors(mol), dtype=np.float32)
            # Descriptor değerleri hesaplanır.
            rows.append(np.nan_to_num(desc, nan=0.0, posinf=0.0, neginf=0.0))
            # NaN ve sonsuz değerler 0 ile değiştirilir.
    names = [f"RDKit_{name}" for name in descriptor_names]
    # RDKit descriptor isimleri prefix ile kaydedilir.
    return np.vstack(rows), names, valid
    # Feature matrisi, feature isimleri ve valid mask döndürülür.


def smiles_to_avalon(smiles):
    """Avalon fingerprint üretir; RDKit kurulumu desteklemiyorsa atlar."""
    try:
        # Avalon modülü her RDKit kurulumunda aktif olmayabilir.
        from rdkit.Avalon import pyAvalonTools
    except Exception:
        # Avalon yoksa workflow durdurulmaz.
        print("Avalon desteği bulunamadı; Avalon fingerprint atlandı.")
        return None, [], None

    mols, valid = valid_molecules(smiles)
    # SMILES listesi molekül nesnelerine çevrilir.
    rows = []
    # Avalon satırları burada biriktirilir.
    for mol, keep in zip(mols, valid):
        # Moleküller geçerlilik bilgisiyle birlikte dolaşılır.
        if keep:
            # Geçersiz SMILES atlanır.
            fp = pyAvalonTools.GetAvalonFP(mol, nBits=1024)
            # Avalon fingerprint hesaplanır.
            arr = np.zeros((1024,), dtype=np.float32)
            # Fingerprint için boş array oluşturulur.
            DataStructs.ConvertToNumpyArray(fp, arr)
            # Avalon fingerprint numpy array'e çevrilir.
            rows.append(arr)
            # Array sonuç listesine eklenir.
    names = [f"Avalon_{i}" for i in range(1024)]
    # Avalon feature isimleri oluşturulur.
    return np.vstack(rows), names, valid
    # Feature matrisi, feature isimleri ve valid mask döndürülür.

## 4. Ham veriden feature CSV üretimi

Her iki veri seti okunur, temizlenir, feature matrisi oluşturulur ve CSV olarak kaydedilir.

In [ ]:
index_rows = []
# Üretilen feature dosyalarının özeti bu listede tutulur.

for dataset_key, info in DATASETS.items():
    # İki veri seti sırayla işlenir.
    print("\n" + "=" * 90)
    # Çıktıda okunabilir ayırıcı çizgi oluşturulur.
    print(f"{dataset_key} işleniyor")
    # Hangi veri setinin işlendiği yazdırılır.
    print("=" * 90)
    # Çıktıda okunabilir ayırıcı çizgi oluşturulur.

    raw_df, sep = read_csv_flexible(info["raw_url"])
    # Ham CSV GitHub raw linkinden okunur.
    print(f"Ham veri: {raw_df.shape[0]} satır, {raw_df.shape[1]} kolon, ayraç={repr(sep)}")
    # Ham verinin boyutu ve ayıracı yazdırılır.

    target_numeric = pd.to_numeric(raw_df[TARGET_COLUMN], errors="coerce")
    # Target kolonu sayısal değere çevrilir.
    usable_mask = target_numeric.notna() & raw_df[SMILES_COLUMN].notna()
    # Target ve SMILES bilgisi dolu olan satırlar seçilir.
    clean_df = raw_df.loc[usable_mask].copy().reset_index(drop=True)
    # Kullanılabilir satırlardan temiz tablo oluşturulur.
    y = pd.to_numeric(clean_df[TARGET_COLUMN], errors="coerce").astype(int).to_numpy()
    # Target numpy integer vektörüne çevrilir.

    print(f"Kullanılabilir satır: {len(clean_df)}")
    # Temiz satır sayısı yazdırılır.
    print(f"Çıkarılan satır: {len(raw_df) - len(clean_df)}")
    # Temizleme sırasında çıkarılan satır sayısı yazdırılır.
    print("Sınıf dağılımı:", dict(pd.Series(y).value_counts().sort_index()))
    # Target sınıf dağılımı yazdırılır.

    smiles = clean_df[SMILES_COLUMN].tolist()
    # Feature üretimi için SMILES listesi alınır.

    X_morgan, names_morgan, valid_morgan = smiles_to_morgan(smiles)
    # Morgan fingerprint matrisi üretilir.
    X_maccs, names_maccs, valid_maccs = smiles_to_maccs(smiles)
    # MACCS fingerprint matrisi üretilir.
    X_rdkit, names_rdkit, valid_rdkit = smiles_to_rdkit_descriptors(smiles)
    # RDKit descriptor matrisi üretilir.
    X_avalon, names_avalon, valid_avalon = smiles_to_avalon(smiles)
    # Avalon destekleniyorsa Avalon fingerprint matrisi üretilir.

    if not (np.array_equal(valid_morgan, valid_maccs) and np.array_equal(valid_morgan, valid_rdkit)):
        # Temel feature bloklarının aynı molekülleri tuttuğu kontrol edilir.
        raise ValueError("Feature blokları arasında geçerli SMILES maskesi farklı çıktı.")

    valid = valid_morgan
    # Ortak geçerli molekül maskesi belirlenir.
    valid_df = clean_df.loc[valid].reset_index(drop=True)
    # Geçerli SMILES içeren satırlar seçilir.
    valid_y = y[valid]
    # Target vektörü aynı maskeyle filtrelenir.

    feature_blocks = [X_morgan, X_maccs, X_rdkit]
    # Birleştirilecek temel feature blokları listelenir.
    feature_names = names_morgan + names_maccs + names_rdkit
    # Temel feature isimleri birleştirilir.

    if X_avalon is not None:
        # Avalon üretildiyse feature tablosuna eklenir.
        if not np.array_equal(valid, valid_avalon):
            # Avalon maskesi diğer bloklarla uyumlu mu kontrol edilir.
            raise ValueError("Avalon geçerli SMILES maskesi diğer feature bloklarıyla uyumsuz.")
        feature_blocks.append(X_avalon)
        # Avalon matrisi feature bloklarına eklenir.
        feature_names += names_avalon
        # Avalon feature isimleri listeye eklenir.

    X_all = np.hstack(feature_blocks)
    # Tüm feature blokları yan yana birleştirilir.

    feature_df = pd.concat(
        [
            pd.DataFrame({
                "Dataset": dataset_key,
                "SMILES": valid_df[SMILES_COLUMN].values,
                "Target": valid_y,
            }),
            # Modelleme için gerekli metadata kolonları hazırlanır.
            pd.DataFrame(X_all, columns=feature_names),
            # Sayısal feature matrisi tabloya çevrilir.
        ],
        axis=1,
    )
    # Metadata ve feature matrisi tek CSV tablosunda birleştirilir.

    out_csv = feature_store_dir / f"{info['model_prefix']}_features.csv"
    # Feature CSV dosyasının kayıt yolu belirlenir.
    feature_df.to_csv(out_csv, index=False)
    # Feature tablosu CSV olarak kaydedilir.

    manifest_csv = feature_store_dir / f"{info['model_prefix']}_feature_manifest.csv"
    # Feature isimlerini tutacak manifest dosyası belirlenir.
    pd.DataFrame({"Feature": feature_names}).to_csv(manifest_csv, index=False)
    # Feature isimleri ayrı CSV olarak kaydedilir.

    counts = feature_df["Target"].value_counts().sort_index()
    # Sınıf dağılımı hesaplanır.
    plt.figure(figsize=(5, 4))
    # Grafik alanı oluşturulur.
    plt.bar([str(i) for i in counts.index], counts.values)
    # Sınıf dağılımı bar chart olarak çizilir.
    plt.xlabel("Sınıf")
    # X ekseni etiketi eklenir.
    plt.ylabel("Molekül sayısı")
    # Y ekseni etiketi eklenir.
    plt.title(f"{info['short_name']} sınıf dağılımı")
    # Grafik başlığı eklenir.
    plt.tight_layout()
    # Yerleşim düzenlenir.
    plt.savefig(feature_store_dir / f"{info['model_prefix']}_class_distribution.png", dpi=300, bbox_inches="tight")
    # Sınıf dağılımı grafiği PNG olarak kaydedilir.
    plt.show()
    # Grafik ekranda gösterilir.

    index_rows.append({
        "Dataset": dataset_key,
        "FeatureFile": str(out_csv),
        "ManifestFile": str(manifest_csv),
        "Rows": feature_df.shape[0],
        "Columns": feature_df.shape[1],
        "FeatureCount": len(feature_names),
        "Class0": int(counts.get(0, 0)),
        "Class1": int(counts.get(1, 0)),
        "HasAvalon": any(name.startswith("Avalon_") for name in feature_names),
    })
    # Feature store özetine bu veri setinin bilgileri eklenir.

feature_index = pd.DataFrame(index_rows)
# Feature store özeti tabloya çevrilir.
feature_index.to_csv(feature_store_dir / "feature_store_index.csv", index=False)
# Feature store özeti CSV olarak kaydedilir.

show_table(feature_index, title="Üretilen feature store özeti")
# Üretilen feature dosyalarının özeti ekranda gösterilir.

## 5. GitHub'a yüklenecek dosyalar

Aşağıdaki liste, `molfest_outputs/01_feature_store/` klasöründe oluşan dosyaları gösterir.

In [ ]:
for path in sorted(feature_store_dir.glob("*")):
    # Feature store klasöründeki dosyalar sırayla dolaşılır.
    size_mb = path.stat().st_size / (1024 * 1024)
    # Dosya boyutu MB olarak hesaplanır.
    print(f"- {path.name} | {size_mb:.2f} MB")
    # Dosya adı ve boyutu ekrana yazdırılır.